In [ ]:
import matplotlib.pyplot as plt
plt.style.use("utils/RESS.mplstyle")

# Figure 2

Sabater 1d test function

In [ ]:
import numpy as np
import torch.distributions as dist
import matplotlib.pyplot as plt
seed_value = 33
np.random.seed(seed_value)

def benchmark_1d_sabater(d, x, a = 2, b = 8):

    return 3 - 4*np.exp(-4*(d-4)**2)-5.2*np.exp(-4*(d-6)**2) + (d-a)/(b - a)*x[:,0] + (b-d)/(b-a)*x[:,1]

def sabater_1d(x1, x2):
    global x_design, count
    count += 1
    a = 2
    b = 8
    Y1 = 3 - 4*np.exp(-4*(x_design-4)**2)-5.2*np.exp(-4*(x_design-6)**2) + (x_design-a)/(b - a)*x1 + (b-x_design)/(b-a)*x2
    
    return Y1

def comp_quant(d, alpha = 0.95, n_samples = 1000):
    
    X1 = np.random.uniform(0.0, 10.0, n_samples)
    X2 = np.random.normal(1.01, 0.71, n_samples)

    X = np.concatenate((X1.reshape(-1,1), X2.reshape(-1,1)), axis = 1)
    y = benchmark_1d_sabater(d, X)

    samples_sorted = np.sort(y, axis = 0)
    quant_idx = int(np.floor(n_samples*alpha))-1
    quantile = samples_sorted[quant_idx]

    # print(X[sorted_idx[quant_idx],:])

    return quantile


n = 100000

ref_d_vec = np.linspace(2.0, 8.0, 1000)
ref_q_vec_8 = np.zeros(len(ref_d_vec))
ref_q_vec_9 = np.zeros(len(ref_d_vec))
ref_q_vec_95 = np.zeros(len(ref_d_vec))
ref_q_vec_5 = np.zeros(len(ref_d_vec))
alpha = 0.8

for idx, d in enumerate(ref_d_vec):
    # print(d)
    ref_q_vec_8[idx] = comp_quant(d, alpha= alpha, n_samples = n)
    ref_q_vec_9[idx] = comp_quant(d, alpha= 0.9, n_samples = n)
    ref_q_vec_95[idx] = comp_quant(d, alpha= 0.95, n_samples = n)
    ref_q_vec_5[idx] = comp_quant(d, alpha= 0.5, n_samples = n)

# fig, ax = plt.subplots()
# ax.plot(ref_d_vec, ref_q_vec_8, label = f"q_{alpha}")
# ax.legend()
# ax.set_xlabel("d")
# ax.set_ylabel("M(d)")
# plt.show()

# print(comp_quant(d = 6.5, alpha= alpha, n_samples = 10000))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

c_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

upper_bound_6 = ref_q_vec_5 + (ref_q_vec_8 - ref_q_vec_5)
lower_bound_6 = ref_q_vec_5 - (ref_q_vec_8 - ref_q_vec_5)
upper_bound_8 = ref_q_vec_5 + (ref_q_vec_9 - ref_q_vec_5)
lower_bound_8 = ref_q_vec_5 - (ref_q_vec_9 - ref_q_vec_5)
upper_bound_9 = ref_q_vec_5 + (ref_q_vec_95 - ref_q_vec_5)
lower_bound_9 = ref_q_vec_5 - (ref_q_vec_95 - ref_q_vec_5)

fig, ax = plt.subplots(figsize = np.array([300, 150])/72.0)
h2 = ax.fill_between(ref_d_vec, upper_bound_9, lower_bound_9, color='lightgray', alpha=0.5, label='$90 \%$')
h2 = ax.fill_between(ref_d_vec, upper_bound_8, lower_bound_8, color='gray', alpha=0.5, label='$80 \%$')
h2 = ax.fill_between(ref_d_vec, upper_bound_6, lower_bound_6, color='dimgray', alpha=0.5, label='$60 \%$')
h1 = ax.plot(ref_d_vec, ref_q_vec_5, label = "Median", color = "Black")
h1 = ax.plot(ref_d_vec, ref_q_vec_8, label = "$q_{0.8}$", color = c_cycle[3], linewidth = 1.5)


ax.legend(loc = "upper left", ncol=2, frameon = False)
ax.set_xlabel("$x$")
ax.set_ylabel("$\mathcal{M}(x,\\xi)$")
ax.set_yticks([0, 5, 10])
sns.despine()

plot_folder = "_plots/RESS"
file_name = "sabater_test_fun1d"
fig.tight_layout()
fig.savefig(f'{plot_folder}/{file_name}.pdf', format = 'pdf', bbox_inches = 'tight', pad_inches = 0.0)

# Figure 3

Snapshots of iterations

In [ ]:
# Load the vars
import pickle
from utils.filemanager import create_result_folder
import numpy as np 
import torch 

# data_file = "_experiments/Sabater1d/illustrative_example/doe_7_data.pkl"
data_file = "_experiments/RESS/sabater1d_demo/doe_10_rep_8_data.pkl"

loaded_objects = []
with open(f'{data_file}', 'rb') as f:
    try:
        while True:
            loaded_objects.append(pickle.load(f))
    except EOFError:
        pass

ro_data, model, gp_model, args = loaded_objects
ref_x = torch.tensor([3.9646])
ref_q = 2.3364

data = torch.load("_reference_comp/Sabater1d_tau0p95.pt", weights_only=False)
ref_x_vec = data.domain_plot['x'].nodes
ref_q_vec = data.domain_plot['J']

In [ ]:
# Plot Iterations
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

ccycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

# Create a figure
fig = plt.figure(figsize=np.array([300, 300])/72.0)

# Define the grid layout
gs = gridspec.GridSpec(4, 2, height_ratios=[1, 1/3, 1, 1/3])

# Create subplots
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[0, 1])
ax4 = fig.add_subplot(gs[1, 1])
ax5 = fig.add_subplot(gs[2, 0])
ax6 = fig.add_subplot(gs[3, 0])
ax7 = fig.add_subplot(gs[2, 1])
ax8 = fig.add_subplot(gs[3, 1])

ax = [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8]

alpha = 0.8
cand_x = torch.hstack(ro_data.EI_infill["cand_x"])


# iter_plts = [0, 3, 5, 7]
iter_plts = [0, 2, 4, 9]
# iter_plts = [4, 5, 6, 7]
# iter_plts = [6, 7, 8, 9]


for i, iter in enumerate(iter_plts):
    idx = i*2

    plt_data = ro_data.EI_infill["plot_data"][iter_plts[i]]
    n_doe = ro_data.doe["n"]
    # ax[idx] = plot_quant(data["x_vec"], data["quant_vec"][:,0], data["quant_mean_vec"][:,0], data["quant_std_vec"][:,0], data["ref_x_vec"], data["ref_q_vec"], data["alpha"], ax = ax[idx], fill = True, leg = False)
    
    h1 = ax[idx].plot(ref_x_vec, ref_q_vec, label = f"q_{alpha}", color = "dimgray", zorder = 1)

    upper_bound = plt_data["quant_mean_vec"][:,0] + plt_data["quant_std_vec"][:,0]
    lower_bound = plt_data["quant_mean_vec"][:,0] - plt_data["quant_std_vec"][:,0]
    h3 = ax[idx].fill_between(plt_data["x_domain"].nodes[:,0], upper_bound, lower_bound, color=ccycle[0], alpha=0.3, zorder = 50)
    h2 = ax[idx].plot(plt_data["x_domain"].nodes, plt_data["quant_mean_vec"][:,0], color=ccycle[0], zorder = 60)
    
    # ax[idx].plot([data["bounds"][0][0], data["bounds"][1][0]], [data["q_star"], data["q_star"]], color = ccycle[3], alpha = 0.8, zorder = 0)
    # ax[idx].scatter(data["x_star"], data["q_star"], marker = "*", color = ccycle[3], label = f"q_min = ({x_star[0,0]:.2f}, {q_star:.2f})", zorder = 100)
    # ax[idx].legend()
    ax[idx].set_title(f"Iteration {iter}")
    ymin, ymax = ax[0].get_ylim()
    h4 = ax[idx].scatter(cand_x[:iter_plts[i]], np.ones(len(cand_x[:iter_plts[i]]))*ymin-0.5, marker = "+", color = "black", linewidth=1, s = 40, label = f"x infill", zorder = 100, clip_on=False)
    ax[idx].set_ylim([ymin, ymax])
    
    h5 = ax[idx+1].plot(plt_data["x_domain"].nodes, plt_data["QEI"].numpy()[:,:5], color = ccycle[2], linestyle='', marker='o', markersize=2, alpha = 0.25)
    h6 = ax[idx+1].scatter(plt_data["x_c"], plt_data["EI_star"], marker = "*", label = "max QEI DiffEvo", color = ccycle[2], zorder = 100, clip_on=False)
    # ax[idx+1].set_ylim([0.0, 2.0])
    # ax[idx+1].legend()

    offset = 5.0

    if i == 0:
        ax[idx].tick_params(bottom = True, labelbottom = False, left = True, labelleft = True)
        ax[idx].set_ylabel("$\mathcal{M}(x)$")
        sns.despine(ax=ax[idx], offset=offset, left = False)
        ax[idx+1].tick_params(bottom = True, labelbottom = False, labelleft = False, left = False)
        ax[idx+1].set_ylabel("$\hat{a}_{\mathrm{EI}}$")
        sns.despine(ax=ax[idx+1], offset=offset, left = False)

    if i == 1:
        ax[idx].tick_params(bottom = True, labelbottom = False, left = False, labelleft = False)
        sns.despine(ax=ax[idx], offset=offset, left = True)
        sns.despine(ax=ax[idx+1], offset=offset, left = True)
        ax[idx+1].tick_params(bottom = True, labelbottom = False, labelleft = False, left = False)
    if i == 2:
        ax[idx].tick_params(bottom = True, labelbottom = False, left = True, labelleft = True)
        sns.despine(ax=ax[idx], offset=offset, left = False)
        sns.despine(ax=ax[idx+1], offset=offset, left = False)
        ax[idx+1].tick_params(bottom = True, labelbottom = True, labelleft = False, left = False)
        ax[idx+1].set_ylabel("$\hat{a}_{\mathrm{EI}}$")
        ax[idx].set_ylabel("$\mathcal{M}(x)$")
        ax[idx+1].set_xlabel("$x$")

    if i == 3:
        ax[idx].tick_params(bottom = True, labelbottom = False, left = False, labelleft = False)
        sns.despine(ax=ax[idx], offset=offset, left = True)
        sns.despine(ax=ax[idx+1], offset=offset, left = True)
        ax[idx+1].tick_params(bottom = True, labelbottom = True, labelleft = False, left = False)
        ax[idx+1].set_xlabel("$x$")


labels = ["$q_{0.8}$", "$\mu_{\\widetilde{Q}_{\\tau}}$", "$\sigma_{\\widetilde{Q}_{\\tau}}$", "infill", "acfn", "max acfn"]
handles = [h1[0], h2[0], h3, h4, h5[0], h6]
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.1), ncol=3, frameon = True)

fig.tight_layout()

In [ ]:
# Save
output_folder = "_plots/RESS"
file_name = 'sabat_1d_iteration'
fig.tight_layout()
fig.savefig(f'{output_folder}/{file_name}.pdf', format = 'pdf', bbox_inches = 'tight', pad_inches = 0.0)

# Figure 4

Evolution of quantile and x error

In [ ]:
# Load the vars
import pickle
from utils.filemanager import create_result_folder
import numpy as np 

data_file = "_experiments/RESS/sabater1d_demo/doe_10_rep_8_data.pkl"

loaded_objects = []
with open(f'{data_file}', 'rb') as f:
    try:
        while True:
            loaded_objects.append(pickle.load(f))
    except EOFError:
        pass

ro_data, model, gp_model, args = loaded_objects

In [ ]:
import numpy as np 
from matplotlib.patches import FancyArrowPatch
import seaborn as sns 
import torch 

def add_double_arrow(ax, start_x, end_x, y_position, label, shift_text_y = 0.02):
    # Create a double arrow to represent the interval
    arrow_length = 0.2  # Adjust this for arrow length
    arrow = FancyArrowPatch((start_x, y_position), (end_x, y_position),
                             arrowstyle='<|-|>', color='black', lw=1,
                             mutation_scale=10, shrinkA= 0.0, shrinkB= 0.0)
    ax.add_patch(arrow)
    
    # Add the label in the middle of the arrow
    ax.text((start_x + end_x) / 2, y_position + shift_text_y, label, ha='center', va='bottom', fontsize=8)

    return ax

def add_iteration_arrows(ax, model_calls_cumsum, n_EI, n_EM, ypos = None, ymin = None, shift_text_y = 0.04):
    ylim = ax.get_ylim()
    delta_y = ylim[1]-ylim[0]
    shift = 0.075

    if ypos is None:
        ypos = ylim[0]-shift*delta_y

    ax = add_double_arrow(ax = ax, start_x=model_calls_cumsum[0], end_x=model_calls_cumsum[n_EI], y_position=ypos, label = "EI", shift_text_y = shift_text_y)
    ax = add_double_arrow(ax = ax, start_x=model_calls_cumsum[n_EI], end_x=model_calls_cumsum[n_EI+n_EM], y_position=ypos, label = "EM", shift_text_y = shift_text_y)
    ax = add_double_arrow(ax = ax, start_x=model_calls_cumsum[n_EI+n_EM], end_x=model_calls_cumsum[-1], y_position=ypos, label = "TR", shift_text_y = shift_text_y)
    
    if ymin is None:
        ax.set_ylim([ylim[0]-1.5*shift*delta_y, ylim[1]])
    else:
        ax.set_ylim([ymin, ylim[1]])
    
    return ax

def plot_best_evolution(
    ro_data,
    model,
    output_folder=None,
    tag=None,
    fig_ax=None,
    continuous_error=True,
):
    
    plot_ref = True

    ref_x = np.array([3.9646])
    ref_q = torch.tensor(2.3364)


    best_x_list = [ro_data.doe["cand_x"]] + ro_data.EI_infill["best_x"] + ro_data.EM_infill["best_x"] + ro_data.TR_infill["best_x"]
    best_q_list = [ro_data.doe["cand_q"]] + ro_data.EI_infill["best_q"] + ro_data.EM_infill["best_q"] + ro_data.TR_infill["best_q"]
    best_q_std_list = [ro_data.doe["cand_q_std"]] + ro_data.EI_infill["best_q_std"] + ro_data.EM_infill["best_q_std"]

    model_calls = ro_data.doe["model_calls"] + ro_data.EI_infill["model_calls"] + ro_data.EM_infill["model_calls"] + ro_data.TR_infill["model_calls"]

    best_x = np.array(best_x_list)
    best_q = np.array(best_q_list)
    best_q_std = np.array(best_q_std_list)
    model_calls_cumsum = np.cumsum(np.array(model_calls))

    n_EI = ro_data.EI_infill["n"]
    n_EM = ro_data.EM_infill["n"]
    n_TR = ro_data.TR_infill["n"]

    created_fig = False
    if fig_ax is None:
        fig, ax = plt.subplots(1, 2, figsize=(6, 3))
        created_fig = True
    else:
        fig, ax = fig_ax

    q_line, = ax[0].plot(model_calls_cumsum, best_q, label="$\\mu_{\\widetilde{Q}_{\\tau}}(x^*)$")

    x_err = model_calls_cumsum[:n_EI + n_EM + 1]
    y_err = best_q[:n_EI + n_EM + 1]
    y_std = best_q_std[:n_EI + n_EM + 1]

    if continuous_error:
        ax[0].fill_between(
            x_err,
            y_err - y_std,
            y_err + y_std,
            alpha=0.3,
            color=q_line.get_color(),
            label="$\\sigma_{\\widetilde{Q}_{\\tau}}(x^*)$"
        )
    else:
        ax[0].errorbar(
            x_err,
            y_err,
            y_std,
            fmt='o',
            capsize=5,
            color=q_line.get_color(),
            label='Std'
        )

    if plot_ref:
        ax[0].plot(
            model_calls_cumsum,
            np.ones(len(model_calls_cumsum)) * ref_q.numpy(),
            linestyle="--",
            color="grey",
            label="Ref. $q$"
        )

    ax[0].set_xlabel("Model calls")
    ax[0].set_yticks([2,4,6])
    ax[0].grid(alpha = 0.3)
    ax[0].set_ylabel("Quantile")
    # ax[0] = add_iteration_arrows(ax[0], model_calls_cumsum, n_EI, n_EM)
    ax[0].legend(
        loc="upper right",
        bbox_to_anchor=(1.05, 1.15),
        frameon = True
    )

    labels = [f"$x_{i}$" for i in range(1, best_x.shape[1] + 1)]

    rel_x_error = np.abs(best_x - ref_x)/np.abs(ref_x)

    # lines = ax[1].plot(model_calls_cumsum, best_x)
    lines = ax[1].semilogy(model_calls_cumsum, rel_x_error, marker = "o", markersize = 2, linewidth = 1.5)
    for line, label in zip(lines, labels):
        line.set_label(label)

    if False:
        for i, line in enumerate(lines):
            ax[1].plot(
                model_calls_cumsum,
                np.ones(len(model_calls_cumsum)) * ref_x[i],
                linestyle="--",
                color=line.get_color()
            )

    ax[1].grid(alpha=0.3)
    ax[1].set_xlabel("Model calls")
    ax[1].set_ylabel(r"$ | x - x^*| / x^* $")
    # ax[1] = add_iteration_arrows(ax[1], model_calls_cumsum, n_EI, n_EM)
    # ax[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0.)

    sns.despine(ax=ax[0])
    sns.despine(ax=ax[1])

    if created_fig:
        fig.tight_layout()

    if output_folder is not None:
        file_name = tag + '_best_evo'
        fig.tight_layout()
        fig.savefig(
            f'{output_folder}/{file_name}.pdf',
            format='pdf',
            bbox_inches='tight',
            pad_inches=0.0
        )

    return fig, ax

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1,2, figsize=np.array([300, 110])/72.0)

x_opt = 3.97

fig, ax = plot_best_evolution(ro_data, model, continuous_error=True, tag = "Opt_evo", fig_ax= (fig, ax))
fig.tight_layout()


fig.text(
    0.025, 0.08, r"\textsf{\textbf{(a)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)

fig.text(
    0.49, 0.08, r"\textsf{\textbf{(b)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)


output_folder = "_plots/RESS"
file_name = 'sabat_1d_evo'
fig.savefig(f'{output_folder}/{file_name}.pdf', format = 'pdf', bbox_inches = 'tight', pad_inches = 0.0)